<a href="https://colab.research.google.com/github/hrittik2002/ML-Notes/blob/main/python/2_Python_Core.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Scope in Python

## 1.1. LEGB — Python's scope lookup chain

When Python sees a name, it searches four scopes in order and stops at the first match. If it finds nothing, you get a NameError.
- **Local** : Inside the current function. Variables assigned inside a function body. Created fresh on each call, destroyed when the call ends.
- **Enclosing** : outer function's scope. Variables in any enclosing def or lambda. Only relevant when functions are nested. This is what makes closures work.
- **Global** : module level. Variables defined at the top of the file (module scope). Shared across all functions in the same module.
- **Built-in** : Python's own names. len, print, range, True, None, exceptions, etc. Always available — searched last.



> Important : LEGB Rule is only for read not for write



In [2]:
# Live example — all four scopes in one snippet

x = "global"              # G — module level

def outer():
    x = "enclosing"        # E — outer function

    def inner():
        x = "local"        # L — inner function
        print(x)           # → "local"      (L wins)
        print(len)        # → built-in fn  (B — found last)
    inner()

    print(x)               # → "enclosing"  (E — local x gone)

outer()
print(x)                   # → "global"     (G)

local
<built-in function len>
enclosing
global


## 1.2. global decleration

- When we declare a variable as global, we just tell the compiler  "when you see this name in this function, treat it as the module-level variable — not a new local."



> Python decides scope at COMPILE TIME, not line by line

### Why you need it ?


In [2]:
x = 100

def fun():
  print(x)
  x = 200

fun()

UnboundLocalError: cannot access local variable 'x' where it is not associated with a value

**Why we are getting *"UnboundLocalError: cannot access local variable 'x' where it is not associated with a value"* here?**

In JS we know the engine does a "hoisting" pass before execution. It scans the function, finds var x, and registers it in the execution context as undefined before any line runs.

In Python — the compiler does a similar pre-scan. It scans the whole function body, sees x = 200 (an assignment), and marks x as local for the entire function — before any line executes.

So when print(x) runs, Python already "knows" x is local (decided at compile time), tries to read it, and it hasn't been assigned yet → UnboundLocalError.

In [3]:
x = 100

def fun():
  global x
  print(x)
  x = 200

fun()

100


global x is just a compile-time instruction to the compiler — "don't treat x as local, look it up at module level."

In [5]:
x = 100

def fun():
  global x
  print(x)
  x = 200
  print(x)

fun()
print(x)

100
200
200


Here is a interesting thing we are actually modifing the global x, x is not becoming local.

In [6]:
# Variable doesn't need to exist at module level before global is used
def init():
    global config, debug
    config = {"debug": True}    # creates it at module level

init()
print(config)   # {'debug': True} — now accessible globally

{'debug': True}


Here we are delaring global to varibales that doesnt even exist.

## 1.3. nonlocal
- it tells compiler "this name belongs to the nearest enclosing function scope — not my own local scope, and not the module"
- When we are just reading the enclosing scope value no need of non-local decleration, but when we are updating any value we must declare it nonlocal

In [8]:
def counter(start):
  count = start

  def increment():
    count += 1

  def decrement():
    count -= 1

  def get_count():
    return count

  return increment, decrement, get_count

increment, decrement, get_count = counter(0)
increment()
increment()
print(get_count())
decrement()
print(get_count())

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value

In [9]:
def counter(start):
  count = start

  def increment():
    nonlocal count
    count += 1

  def decrement():
    nonlocal count
    count -= 1 # whenever we updating the non local value we need to use this

  def get_count():
    return count # here nonlocal declaration is not needed as we are just reading

  return increment, decrement, get_count

increment, decrement, get_count = counter(0)
increment()
increment()
print(get_count())
decrement()
print(get_count())

2
1


**nonlocal cannot create a new variable**

nonlocal x requires that x already exists in some enclosing def or lambda. Python raises SyntaxError: no binding for nonlocal 'x' found otherwise. Unlike global, which can create a new module-level variable.

# 2. Closure

Accoring to the concept of Closure, when a outer function returns a inner funtion, outer function's frame would normally be destroyed but inner function holds a reference to it. Frame stays alive and inner can use those variables in the future


In [10]:
# Python
def multiplier(factor):       # factor lives here
    def multiply(n):
        return n * factor     # inner holds reference to factor
    return multiply           # outer is done, but factor survives

double = multiplier(2)        # outer finished, factor=2 still alive
double(5)                     # 10 — factor=2 is still there

10

When Python compiles the function, it notices that multiply uses factor but factor isn't defined inside multiply. So it marks factor as a free variable — meaning "this comes from outside."

At this point Python creates a cell object — a special box that holds the value.
Both the outer frame and the inner function point to the same cell — not a copy, the same box.

When multiplier(2) returns, the frame is destroyed — but the cell object survives because multiply.__closure__ still holds a reference to it.

In [16]:
double.__closure__[0].cell_contents   # 2  ← the cell object, value inside
double.__code__.co_freevars           # ('factor',) ← name of the free vari

('factor',)

Python wraps outer variables into cell objects at compile time.
The inner function holds a reference to those cells.
Cells survive as long as the inner function exists.